# Querying the live ADS-B database over Quack

This notebook connects a **separate** DuckDB instance to the database that
`adsb-data-engine` is actively recording into, and plots a histogram of
aircraft seen per day.

## Why this exists

Embedded DuckDB takes an **exclusive file lock**. While the desktop app is
running it owns `adsb_history.db`, so anything else that opens the file is
locked out — which is why the older `duckdb_storage_browser.ipynb` has to
point at the file directly and only works when the app is closed.

[Quack](https://duckdb.org/docs/current/quack/overview) turns the running
instance into an HTTP server that other DuckDB clients can `ATTACH` to. The
engine keeps sole ownership and stays the only writer; this notebook is just
another client. **The app can keep recording the whole time.**

## Before you start

1. In the desktop app, click **Share DB** in the metrics bar. It copies a
   ready-to-paste `ATTACH` statement — including the token — to your clipboard.
2. Paste the token into the cell below.

> **The token is a credential.** Anyone holding it has full read *and write*
> access to every table, because the server runs with Quack's default
> permissive authorization. Don't commit it, and don't share the notebook with
> the token filled in.

> **Quack is beta** until DuckDB 2.0. Keep the client and server on the same
> DuckDB version — the protocol is still allowed to change between releases.


In [ ]:
# Quack needs DuckDB 1.5.3+. Match the engine's version (currently v1.5.5) —
# client and server run the same beta protocol.
%pip install 'duckdb>=1.5.5' pandas matplotlib

import duckdb
import pandas as pd
import matplotlib.pyplot as plt


## Connection details

`QUACK_URI` is whatever the app reported when you enabled sharing. The default
bind is `quack:localhost` (port 9494); the server refuses non-local hostnames
unless it was started with `allow_other_hostname`.


In [ ]:
QUACK_URI = "quack:localhost:9494"
TOKEN = "REDACTED-QUACK-TOKEN"  # paste from the app's Share DB dialog

if not TOKEN:
    raise ValueError(
        "No token set. Click 'Share DB' in the desktop app and paste the token here."
    )


In [ ]:
con = duckdb.connect()  # a throwaway in-memory instance; the data lives remotely
print(con.execute("SELECT version()").fetchone()[0])

# `quack` is not statically linked into most builds — this fetches it from
# extensions.duckdb.org the first time, so it needs network access once.
con.execute("INSTALL quack")
con.execute("LOAD quack")


In [ ]:
# Bind the token as a parameter rather than formatting it into the SQL, so it
# does not end up in query logs or a traceback.
con.execute(f"ATTACH '{QUACK_URI}' AS adsb (TOKEN ?)", [TOKEN])
print(f"attached to {QUACK_URI}")


## What's in there

Data queries work as if the tables were local — `adsb.positions`, `adsb.flights`
and the rest are ordinary tables you can join against.

**Metadata is the exception.** `adsb.information_schema` does not exist, and
`SHOW TABLES FROM adsb` / `duckdb_tables()` return **zero rows without erroring**
— they only inspect the local instance. Use the attached catalog's `query()`
macro to run the metadata query *on the server* instead.


In [ ]:
con.execute("""
    SELECT * FROM adsb.query('
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = ''main''
        ORDER BY table_name
    ')
""").df()


In [ ]:
con.execute("""
    SELECT
        count(*)                            AS positions,
        count(DISTINCT hex_ident)           AS aircraft,
        epoch_ms(min(timestamp_ms))         AS first_seen_utc,
        epoch_ms(max(timestamp_ms))         AS last_seen_utc
    FROM adsb.positions
""").df()


## Histogram: aircraft per day

`timestamp_ms` is epoch milliseconds (`BIGINT`), so `epoch_ms()` converts it to
a UTC `TIMESTAMP`. Prefer it over dividing by 1000 — DuckDB's `/` is float
division, and rounding on a timestamp boundary silently shifts rows into the
wrong bucket.

Aggregation runs **on the server** and only the grouped rows come back, so this
stays cheap even against millions of positions.


In [ ]:
daily = con.execute("""
    SELECT
        CAST(epoch_ms(timestamp_ms) AS DATE) AS day,
        count(DISTINCT hex_ident)            AS aircraft,
        count(*)                             AS positions
    FROM adsb.positions
    GROUP BY day
    ORDER BY day
""").df()

daily


In [ ]:
if daily.empty:
    print("No positions recorded yet — start the feed and re-run.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(daily["day"], daily["aircraft"], color="#38bdf8", edgecolor="none")
    ax.set_title("Distinct aircraft seen per day")
    ax.set_xlabel("day (UTC)")
    ax.set_ylabel("aircraft")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## Track history: flights per day

`positions` is raw samples. The `flights` table is the engine's **track
segmentation** — it splits one aircraft's positions into separate flights
whenever there is a gap longer than `gap_threshold_ms` (1 hour by default), so
an aircraft seen on three passes counts as three flights, not one.

That makes it the better basis for "how much traffic did we actually track".


In [ ]:
flights = con.execute("""
    SELECT
        CAST(epoch_ms(first_seen_ms) AS DATE) AS day,
        count(*)                              AS flights,
        count(DISTINCT hex_ident)             AS aircraft,
        round(avg(position_count), 1)         AS avg_positions_per_flight,
        round(max(max_altitude))              AS max_altitude_ft
    FROM adsb.flights
    GROUP BY day
    ORDER BY day
""").df()

flights


In [ ]:
if flights.empty:
    print("No flights recorded yet.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(flights["day"], flights["flights"], color="#a78bfa",
           edgecolor="none", label="flights")
    ax.plot(flights["day"], flights["aircraft"], color="#f59e0b",
            marker="o", linewidth=2, label="distinct aircraft")
    ax.set_title("Tracked flights per day (vs distinct airframes)")
    ax.set_xlabel("day (UTC)")
    ax.set_ylabel("count")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## It really is live

Re-run this while the feed is running and the count moves — you are reading the
same database the app is writing to, not a snapshot or an export.


In [ ]:
con.execute("""
    SELECT
        count(*)                    AS positions_last_5_min,
        count(DISTINCT hex_ident)   AS aircraft_last_5_min
    FROM adsb.positions
    WHERE timestamp_ms > (epoch_ms(now()) - 5 * 60 * 1000)
""").df()


## Disconnect

Detaching only closes this client. The engine keeps serving and keeps
recording; use the app's **Shared** button to stop sharing altogether.


In [ ]:
con.execute("DETACH adsb")
con.close()
print("detached")
